# E023 — Multimodal fusion: LPCC audio + image

Replicates the E009 calibrated score-fusion pipeline but swaps E008 MFCC audio
for E020 LPCC audio. Both modalities use their +All augmentation flagships:

- **Audio:** LPCC 13+Δ+ΔΔ+CMN (39d), UBM-32 + MAP r=16, +All aug (noise SNR=20 + speed ±10%)
- **Image:** PCA 50 + LogReg C=1, +All aug (flip + brightness[0.7,1.3] + noise σ=15)

Steps:
1. LOSO 3-fold: collect OOF LPCC audio scores + OOF image scores independently
2. Platt calibration on OOF for each modality
3. Grid search w∈[0,1] for fused = w·audio_cal + (1−w)·image_cal
4. Compare vs E009 (MFCC+image, grid w=0.28, OOF EER=3.75%, min-DCF=0.0750)

**Reference (E009):** OOF EER = 3.75%, min-DCF = 0.0750, optimal w = 0.28

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "audio":     "#E67E22",
    "image":     "#8E44AD",
    "fusion":    "#E74C3C",
    "e009":      "#2E86AB",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67

print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. Audio pipeline — LPCC +All

Identical to E020. UBM-32 + MAP r=16. +All augmentation (noise SNR=20dB + speed ±10%).
Val utterances scored from original WAVs only.

In [ ]:
# --- Audio helpers ---

def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def extract_lpcc(y: np.ndarray, sr: int, order: int = 12, n_cep: int = 13,
                 hop_length: int = 160, win_length: int = 400) -> np.ndarray:
    """Extract LPCC+Δ+ΔΔ+CMN. Returns (T, 39)."""
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    lpcc_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        lpcc_frames.append(cep)
    lpcc   = np.array(lpcc_frames, dtype=np.float32)
    delta  = librosa.feature.delta(lpcc.T).T
    delta2 = librosa.feature.delta(lpcc.T, order=2).T
    feat   = np.hstack([lpcc, delta, delta2])
    feat  -= feat.mean(axis=0)  # CMN
    return feat


def aug_noise_audio(y: np.ndarray, rng: np.random.Generator, snr_db: float = 20.0) -> np.ndarray:
    p = np.mean(y ** 2) + 1e-10
    return y + rng.normal(0, np.sqrt(p / 10 ** (snr_db / 10)), len(y)).astype(y.dtype)


def aug_speed(y: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    return librosa.effects.time_stretch(y, rate=rng.uniform(0.9, 1.1))


def load_and_augment_audio(wav_path: Path, rng: np.random.Generator):
    """Return [(y, sr), ...] — original + noise + speed."""
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    return [
        (y, sr),
        (aug_noise_audio(y, rng), sr),
        (aug_speed(y, rng), sr),
    ]


def extract_audio_batch(df: pd.DataFrame, data_dir: Path, seed: int):
    """Extract LPCC features for all samples with +All aug."""
    rng = np.random.default_rng(seed)
    all_feat, all_labels = [], []
    for _, row in df.iterrows():
        for y_aug, sr in load_and_augment_audio(find_wav(row["stem"], data_dir), rng):
            feat = extract_lpcc(y_aug, sr)
            all_feat.append(feat)
            all_labels.extend([row["label"]] * len(feat))
    return np.vstack(all_feat), np.array(all_labels)


def score_utterance_lpcc(wav_path: Path, adapted: GaussianMixture, ubm: GaussianMixture) -> float:
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat  = extract_lpcc(y, sr)
    return float((adapted.score_samples(feat) - ubm.score_samples(feat)).mean())


def train_ubm(X: np.ndarray, n_components: int = 32, seed: int = 67) -> GaussianMixture:
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm: GaussianMixture, X_target: np.ndarray, r: float = 16.0) -> GaussianMixture:
    log_prob  = ubm._estimate_log_prob(X_target)
    log_resp  = log_prob + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp      = np.exp(log_resp)
    n_k       = resp.sum(axis=0)
    mu_hat    = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha     = n_k / (n_k + r)
    adapted   = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


print("Audio (LPCC) pipeline functions defined.")

## 2. Image pipeline — PCA 50 + LogReg +All

Identical to E007 +All. Val images: originals only.

In [ ]:
# --- Image helpers ---

def find_png(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".png")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def load_image(png_path: Path) -> np.ndarray:
    """Load PNG → grayscale → flatten. Returns (6400,)."""
    img = Image.open(png_path).convert("RGB")
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2).flatten()


def load_images(df: pd.DataFrame, data_dir: Path) -> np.ndarray:
    return np.stack([load_image(find_png(row["stem"], data_dir)) for _, row in df.iterrows()])


def aug_flip(x: np.ndarray) -> np.ndarray:
    return x.reshape(80, 80)[:, ::-1].flatten()


def aug_brightness(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    return np.clip(x * rng.uniform(0.7, 1.3), 0, 255)


def aug_noise_img(x: np.ndarray, rng: np.random.Generator, sigma: float = 15.0) -> np.ndarray:
    return np.clip(x + rng.normal(0, sigma, x.shape), 0, 255)


def augment_images(X: np.ndarray, y: np.ndarray, seed: int) -> tuple:
    """Apply +All image aug: original + flip + brightness + noise."""
    rng = np.random.default_rng(seed)
    aug_X, aug_y = [], []
    for xi, yi in zip(X, y):
        aug_X.extend([aug_flip(xi), aug_brightness(xi, rng), aug_noise_img(xi, rng)])
        aug_y.extend([yi, yi, yi])
    return np.vstack([X, np.stack(aug_X)]), np.concatenate([y, np.array(aug_y)])


print("Image pipeline functions defined.")

## 3. LOSO CV — collect OOF scores for both modalities

Both modalities share the same folds. For each fold:
- Audio: train UBM on non-target LPCC frames, MAP-adapt on target LPCC frames, score val WAVs
- Image: train StandardScaler+PCA+LogReg on augmented train images, score val PNGs

Val data is never augmented.

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0
N_PCA = 50
C_LOGREG = 1.0

oof_audio = np.full(len(manifest), np.nan)
oof_image = np.full(len(manifest), np.nan)

audio_fold_results = []
image_fold_results = []

print("Running LOSO CV — collecting OOF for LPCC audio + image")
print("=" * 60)

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]
    y_train  = train_df["label"].to_numpy()
    y_val    = val_df["label"].to_numpy()

    print(f"\nFold {fold_id}: {len(train_df)} train, {len(val_df)} val")

    # ── Audio ─────────────────────────────────────────────────────────────
    X_audio_aug, y_audio_aug = extract_audio_batch(train_df, DATA, seed=SEED + fold_id)
    X_nt = X_audio_aug[y_audio_aug == 0]
    X_t  = X_audio_aug[y_audio_aug == 1]
    ubm     = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)
    adapted = map_adapt(ubm, X_t, r=MAP_R)

    for idx, row in val_df.iterrows():
        oof_audio[idx] = score_utterance_lpcc(find_wav(row["stem"], DATA), adapted, ubm)

    audio_val  = oof_audio[val_idx]
    eer_a, _   = compute_eer(audio_val[y_val==1], audio_val[y_val==0])
    dcf_a, _   = compute_min_dcf(audio_val[y_val==1], audio_val[y_val==0])
    audio_fold_results.append({"fold": fold_id, "eer": eer_a, "min_dcf": dcf_a})
    print(f"  Audio  EER={eer_a*100:.2f}%  min-DCF={dcf_a:.4f}")

    # ── Image ─────────────────────────────────────────────────────────────
    X_train_orig = load_images(train_df, DATA)
    X_val_orig   = load_images(val_df,   DATA)
    X_train_aug, y_train_aug = augment_images(X_train_orig, y_train, seed=SEED + fold_id)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train_aug)
    X_vl_s = scaler.transform(X_val_orig)

    pca = PCA(n_components=N_PCA, random_state=SEED)
    X_tr_p = pca.fit_transform(X_tr_s)
    X_vl_p = pca.transform(X_vl_s)

    clf = LogisticRegression(C=C_LOGREG, max_iter=1000, random_state=SEED)
    clf.fit(X_tr_p, y_train_aug)
    oof_image[val_idx] = clf.decision_function(X_vl_p)

    image_val = oof_image[val_idx]
    eer_i, _  = compute_eer(image_val[y_val==1], image_val[y_val==0])
    dcf_i, _  = compute_min_dcf(image_val[y_val==1], image_val[y_val==0])
    image_fold_results.append({"fold": fold_id, "eer": eer_i, "min_dcf": dcf_i})
    print(f"  Image  EER={eer_i*100:.2f}%  min-DCF={dcf_i:.4f}")

print("\n" + "=" * 60)
print("OOF collection complete.")

# Per-fold summary
audio_eers = [r["eer"]*100 for r in audio_fold_results]
image_eers = [r["eer"]*100 for r in image_fold_results]
print(f"\nAudio LPCC per-fold EER: {audio_eers[0]:.2f}, {audio_eers[1]:.2f}, {audio_eers[2]:.2f}  "
      f"mean={np.mean(audio_eers):.2f}±{np.std(audio_eers):.2f}%")
print(f"Image +All per-fold EER: {image_eers[0]:.2f}, {image_eers[1]:.2f}, {image_eers[2]:.2f}  "
      f"mean={np.mean(image_eers):.2f}±{np.std(image_eers):.2f}%")

## 4. Platt calibration

Fit LogisticRegression (C=1e6, class_weight='balanced') on OOF scores of each
modality separately. The calibrated scores are log-odds on a common scale,
enabling meaningful linear combination.

In [ ]:
def platt_calibrate(scores: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Fit Platt scaling on (scores, labels), return calibrated log-odds."""
    cal = LogisticRegression(C=1e6, max_iter=1000, class_weight="balanced", random_state=SEED)
    cal.fit(scores.reshape(-1, 1), labels)
    return cal.decision_function(scores.reshape(-1, 1))


oof_audio_cal = platt_calibrate(oof_audio, y_all)
oof_image_cal = platt_calibrate(oof_image, y_all)

print("Calibration complete.")
print(f"Audio calibrated range: [{oof_audio_cal.min():.3f}, {oof_audio_cal.max():.3f}]")
print(f"Image calibrated range: [{oof_image_cal.min():.3f}, {oof_image_cal.max():.3f}]")

# OOF EER before/after calibration
eer_a_raw, _ = compute_eer(oof_audio[y_all==1], oof_audio[y_all==0])
eer_i_raw, _ = compute_eer(oof_image[y_all==1], oof_image[y_all==0])
eer_a_cal, _ = compute_eer(oof_audio_cal[y_all==1], oof_audio_cal[y_all==0])
eer_i_cal, _ = compute_eer(oof_image_cal[y_all==1], oof_image_cal[y_all==0])
print(f"\nOOF EER (unchanged by Platt, monotone transform):")
print(f"  Audio raw={eer_a_raw*100:.2f}%  cal={eer_a_cal*100:.2f}%")
print(f"  Image raw={eer_i_raw*100:.2f}%  cal={eer_i_cal*100:.2f}%")

corr = np.corrcoef(oof_audio_cal, oof_image_cal)[0, 1]
print(f"\nPearson correlation audio_cal vs image_cal: {corr:.3f}")

## 5. Score space visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 2D scatter
ax = axes[0]
ax.scatter(oof_audio_cal[y_all==0], oof_image_cal[y_all==0],
           c=COLORS["nontarget"], alpha=0.4, s=20, label="non-target")
ax.scatter(oof_audio_cal[y_all==1], oof_image_cal[y_all==1],
           c=COLORS["target"], alpha=0.9, s=80, label="target (m431)", zorder=5)
ax.axvline(0, color=COLORS["gray"], ls="--", lw=1, alpha=0.7, label="audio thresh=0")
ax.axhline(0, color=COLORS["gray"], ls=":",  lw=1, alpha=0.7, label="image thresh=0")
ax.set_xlabel("Audio LPCC calibrated score")
ax.set_ylabel("Image calibrated score")
ax.set_title(f"Score space — LPCC audio vs image (calibrated OOF)\nPearson r={corr:.3f}")
ax.legend(fontsize=9)

# Marginal distributions
ax = axes[1]
for scores, name, color, ls in [
    (oof_audio_cal, "Audio LPCC E020", COLORS["audio"], "-"),
    (oof_image_cal, "Image E007",      COLORS["image"], "--"),
]:
    eer_c, _ = compute_eer(scores[y_all==1], scores[y_all==0])
    bins = np.linspace(scores.min(), scores.max(), 40)
    ax.hist(scores[y_all==0], bins=bins, alpha=0.15, color=color, density=True)
    ax.hist(scores[y_all==1], bins=bins, alpha=0.4,  color=color, density=True,
            label=f"{name} (EER={eer_c*100:.1f}%)", histtype="step", lw=2, ls=ls)
ax.axvline(0, color="black", ls="--", lw=1.5, label="threshold = 0")
ax.set_xlabel("Calibrated score")
ax.set_title("Score distributions — both modalities overlaid")
ax.legend(fontsize=9)

plt.suptitle("E023 — Calibrated score space (LPCC audio + image)", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Fusion: average + grid search

fused = w·audio_cal + (1−w)·image_cal

Grid search over w∈[0,1] to find the weight that minimises OOF EER.

In [ ]:
# Average fusion
scores_avg = (oof_audio_cal + oof_image_cal) / 2

# Grid search
weights = np.linspace(0, 1, 101)
grid_eers    = []
grid_min_dcf = []
for w in weights:
    s = w * oof_audio_cal + (1 - w) * oof_image_cal
    eer, _   = compute_eer(s[y_all==1], s[y_all==0])
    dcf, _   = compute_min_dcf(s[y_all==1], s[y_all==0])
    grid_eers.append(eer)
    grid_min_dcf.append(dcf)

best_w_eer = weights[np.argmin(grid_eers)]
best_eer   = min(grid_eers)
best_w_dcf = weights[np.argmin(grid_min_dcf)]
best_dcf   = min(grid_min_dcf)

scores_grid = best_w_eer * oof_audio_cal + (1 - best_w_eer) * oof_image_cal
scores_grid_dcf = best_w_dcf * oof_audio_cal + (1 - best_w_dcf) * oof_image_cal

print(f"Grid search (min EER):    best w={best_w_eer:.2f}, EER={best_eer*100:.2f}%")
print(f"Grid search (min DCF):    best w={best_w_dcf:.2f}, min-DCF={best_dcf:.4f}")
print(f"\nE009 reference: grid w=0.28, EER=3.75%, min-DCF=0.0750")

## 7. Results table — all systems

In [ ]:
systems = [
    ("Audio LPCC (raw)",        oof_audio),
    ("Audio LPCC (cal)",        oof_audio_cal),
    ("Image (raw)",             oof_image),
    ("Image (cal)",             oof_image_cal),
    ("Average (w=0.50)",        scores_avg),
    (f"Grid EER (w={best_w_eer:.2f})", scores_grid),
    (f"Grid DCF (w={best_w_dcf:.2f})", scores_grid_dcf),
]

print(f"{'System':<28} {'OOF EER [%]':>12} {'min-DCF':>10} {'Threshold':>12}")
print("-" * 66)

result_rows = []
for name, scores in systems:
    eer, thr_eer = compute_eer(scores[y_all==1], scores[y_all==0])
    dcf, thr_dcf = compute_min_dcf(scores[y_all==1], scores[y_all==0])
    print(f"{name:<28} {eer*100:>12.2f} {dcf:>10.4f} {thr_dcf:>12.4f}")
    result_rows.append({"name": name, "eer": eer, "min_dcf": dcf,
                        "threshold": thr_dcf, "scores": scores})

print("-" * 66)
print("\nE009 reference (MFCC+image, grid w=0.28): EER=3.75%, min-DCF=0.0750")

# Comparison table requested in task
print("\n" + "=" * 70)
print("Full comparison table:")
print("=" * 70)
comparison = [
    ("Audio MFCC E008",             9.17,  None),
    ("Audio LPCC E020",             6.46,  None),
    ("Image E007",                  4.01,  None),
    ("Fusion MFCC+Image E009 grid", 3.75,  0.0750),
]
# find grid EER fusion result
grid_eer_row = next(r for r in result_rows if "Grid EER" in r["name"])
grid_dcf_row = next(r for r in result_rows if "Grid DCF" in r["name"])
comparison.append((f"Fusion LPCC+Image E023 grid (w={best_w_eer:.2f})",
                   grid_eer_row["eer"]*100, grid_eer_row["min_dcf"]))

print(f"{'System':<45} {'OOF EER [%]':>12} {'min-DCF':>10}")
print("-" * 70)
for name, eer_pct, dcf in comparison:
    dcf_str = f"{dcf:.4f}" if dcf is not None else "—"
    print(f"{name:<45} {eer_pct:>12.2f} {dcf_str:>10}")
print("-" * 70)

delta_eer = grid_eer_row["eer"]*100 - 3.75
direction = "improvement" if delta_eer < 0 else "regression"
print(f"\nE023 vs E009 (grid EER): {delta_eer:+.2f}pp ({direction})")

## 8. Grid search visualization + score distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Grid search curve
ax = axes[0]
ax.plot(weights, [e*100 for e in grid_eers], color=COLORS["fusion"], lw=2, label="EER")
ax.plot(weights, [d*100 for d in grid_min_dcf], color=COLORS["e009"], lw=2, ls="--", label="min-DCF×100")
ax.axvline(best_w_eer, color=COLORS["green"], ls="--", lw=2,
           label=f"best EER w={best_w_eer:.2f} → {best_eer*100:.2f}%")
ax.axvline(0.28, color=COLORS["e009"], ls=":", lw=1.5,
           label="E009 w=0.28 (MFCC+image)")

eer_audio_cal, _ = compute_eer(oof_audio_cal[y_all==1], oof_audio_cal[y_all==0])
eer_image_cal, _ = compute_eer(oof_image_cal[y_all==1], oof_image_cal[y_all==0])
ax.axhline(eer_audio_cal*100, color=COLORS["audio"], ls="--", lw=1, alpha=0.7,
           label=f"audio alone {eer_audio_cal*100:.1f}%")
ax.axhline(eer_image_cal*100, color=COLORS["image"], ls="--", lw=1, alpha=0.7,
           label=f"image alone {eer_image_cal*100:.1f}%")

ax.set_xlabel("Audio weight w  (image weight = 1−w)")
ax.set_ylabel("EER [%] / min-DCF×100")
ax.set_title("Grid search: optimal audio/image weight\nLPCC audio (E023) vs MFCC (E009 ref)")
ax.legend(fontsize=8)

# Score distribution for best grid fusion
ax = axes[1]
best_scores = scores_grid
best_eer_val, best_thr = compute_eer(best_scores[y_all==1], best_scores[y_all==0])
bins = np.linspace(best_scores.min(), best_scores.max(), 35)
ax.hist(best_scores[y_all==0], bins=bins, alpha=0.6, color=COLORS["nontarget"],
        label="non-target", density=True)
ax.hist(best_scores[y_all==1], bins=bins, alpha=0.75, color=COLORS["target"],
        label="target", density=True)
ax.axvline(best_thr, color=COLORS["green"], ls="--", lw=2,
           label=f"thresh={best_thr:.3f}")
ax.axvline(0, color=COLORS["gray"], ls=":", lw=1.5, label="ideal=0")
ax.set_xlabel("Fusion score")
ax.set_title(f"Score distribution — Grid EER (w={best_w_eer:.2f})\nEER={best_eer_val*100:.2f}%")
ax.legend(fontsize=8)

plt.suptitle("E023 — Fusion grid search (LPCC audio + image)", fontsize=12)
plt.tight_layout()
plt.show()

## 9. DET curves — E023 vs E009 comparison

In [ ]:
ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos    = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t*100)}" for t in ticks]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: all E023 systems
ax = axes[0]
det_systems = [
    ("Audio LPCC",    oof_audio_cal,  COLORS["audio"],  "--", 1.5),
    ("Image",         oof_image_cal,  COLORS["image"],  "--", 1.5),
    ("Average w=0.5", scores_avg,     COLORS["gray"],   "-",  1.5),
    (f"Grid EER w={best_w_eer:.2f}", scores_grid, COLORS["fusion"], "-", 2.5),
]
for name, scores, color, ls, lw in det_systems:
    fpr, tpr, _ = roc_curve(y_all, scores)
    far_c = np.clip(fpr, 1e-4, 1-1e-4)
    frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
    eer_c, _ = compute_eer(scores[y_all==1], scores[y_all==0])
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c), color=color, lw=lw, ls=ls,
            label=f"{name}  EER={eer_c*100:.1f}%")
ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET curves — E023 systems (OOF)")
ax.legend(fontsize=9)

# Right: E023 best vs E009 reference
ax = axes[1]

# E023 best grid
fpr, tpr, _ = roc_curve(y_all, scores_grid)
far_c = np.clip(fpr, 1e-4, 1-1e-4)
frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
        color=COLORS["fusion"], lw=2.5,
        label=f"E023 LPCC+img grid (w={best_w_eer:.2f})  EER={best_eer*100:.2f}%")

# E009 reference point on diagonal
e009_eer_norm = scipy_norm.ppf(3.75 / 100)
ax.scatter([e009_eer_norm], [e009_eer_norm], marker="x", s=150,
           color=COLORS["e009"], zorder=6, lw=2.5,
           label="E009 MFCC+img grid EER=3.75% (ref, point only)")

# Audio and image alone
for name, scores, color, ls in [
    ("Audio LPCC", oof_audio_cal, COLORS["audio"], "--"),
    ("Image",      oof_image_cal, COLORS["image"], ":"),
]:
    fpr, tpr, _ = roc_curve(y_all, scores)
    far_c = np.clip(fpr, 1e-4, 1-1e-4)
    frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
    eer_c, _ = compute_eer(scores[y_all==1], scores[y_all==0])
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c), color=color, lw=1.5, ls=ls,
            label=f"{name}  EER={eer_c*100:.1f}%")

ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("DET — E023 vs E009 (reference)")
ax.legend(fontsize=8)

plt.suptitle("E023 — DET curve comparison", fontsize=12)
plt.tight_layout()
plt.show()

## 10. Final summary

In [ ]:
grid_r = next(r for r in result_rows if "Grid EER" in r["name"])
avg_r  = next(r for r in result_rows if "Average" in r["name"])

print("E023 — Final summary")
print("=" * 55)
print(f"Audio LPCC per-fold mean EER: {np.mean(audio_eers):.2f} ± {np.std(audio_eers):.2f}%")
print(f"Audio LPCC OOF EER (cal):     {eer_audio_cal*100:.2f}%")
print(f"Image +All per-fold mean EER: {np.mean(image_eers):.2f} ± {np.std(image_eers):.2f}%")
print(f"Image +All OOF EER (cal):     {eer_image_cal*100:.2f}%")
print(f"LPCC–image Pearson corr:      {corr:.3f}")
print()
print(f"Fusion average (w=0.5):       EER={avg_r['eer']*100:.2f}%  min-DCF={avg_r['min_dcf']:.4f}")
print(f"Fusion grid (w={best_w_eer:.2f}):         EER={grid_r['eer']*100:.2f}%  min-DCF={grid_r['min_dcf']:.4f}")
print()
print("Reference — E009 MFCC+image:")
print(f"  Grid (w=0.28): EER=3.75%, min-DCF=0.0750")
print()
delta_eer = grid_r["eer"]*100 - 3.75
delta_dcf = grid_r["min_dcf"] - 0.0750
print(f"E023 vs E009 (grid, EER):  {delta_eer:+.2f}pp ({'improvement' if delta_eer < 0 else 'regression'})")
print(f"E023 vs E009 (grid, DCF):  {delta_dcf:+.4f} ({'improvement' if delta_dcf < 0 else 'regression'})")
print(f"Optimal audio weight: {best_w_eer:.2f} (image weight: {1-best_w_eer:.2f})")